In [9]:
from pathlib import Path
from typing import Optional
from pydantic import BaseModel, Field
import duckdb


def get_project_root() -> Path:
    """Ищет корень проекта, поднимаясь вверх по папкам до тех пор, пока не найдет olist.db"""
    current_path = Path.cwd()
    
    
    for parent in [current_path] + list(current_path.parents):
        if (parent / "olist.db").exists():
            return parent
            
    
    return current_path

DB_PATH = get_project_root() / "olist.db"

class FindOrderArgs(BaseModel):
    """
    Аргументы для поиска заказа в базе данных Olist.
    Используется агентом поддержки, когда клиенту нужно найти информацию 
    по его заказу с помощью email или уникального идентификатора заказа.
    """
    customer_email: Optional[str] = Field(
        default=None, 
        description="Email клиента, по которому нужно найти его заказы в базе."
    )
    order_id: Optional[str] = Field(
        default=None,
        description="Уникальный идентификатор заказа (order_id), если он известен."
    )


def find_order(args: FindOrderArgs) -> str:
    """Выполняет поиск заказа в DuckDB по переданным аргументам"""
    # Подключаемся строго по абсолютному пути к главной базе в корне
    con = duckdb.connect(str(DB_PATH))
    result = []

    try:
        if args.order_id:
            query = """
                SELECT 
                    o.order_id,
                    o.order_status,
                    o.order_purchase_timestamp,
                    c.customer_city,
                    c.customer_email
                FROM olist_orders o 
                JOIN olist_customers c ON o.customer_id = c.customer_id
                WHERE o.order_id = ?
            """
            result = con.execute(query, [args.order_id]).fetchall()
            
        elif args.customer_email:
            query = """
                SELECT 
                    o.order_id,
                    o.order_status,
                    o.order_purchase_timestamp,
                    c.customer_city,
                    c.customer_email
                FROM olist_orders o 
                JOIN olist_customers c ON o.customer_id = c.customer_id
                WHERE c.customer_email = ?
            """
            result = con.execute(query, [args.customer_email]).fetchall()
            
        else:
            return "Уточните order_id или email для поиска заказа."

        if not result:
            return "Заказ не найден в базе данных."

        return str(result)
        
    finally:
        con.close()

In [10]:
find_order(FindOrderArgs(customer_email = 'user_290c77bc52@olist-mail.com'))

"[('29150127e6685892b6eab3eec79f59c7', 'delivered', datetime.datetime(2018, 1, 12, 20, 48, 24), 'sao bernardo do campo', 'user_290c77bc52@olist-mail.com')]"

In [11]:
import os
from openai import OpenAI
from project.skills.skills import FindOrderArgs, find_order

In [12]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "find_order",
            "description": "Поиск информации о заказах клиента по email или ID заказа",
            "parameters": FindOrderArgs.model_json_schema(),
        }
    }
]

response = client.chat.completions.create(
    model="GPT-4o-mini", 
    messages=[
        {"role": "user", "content": "Здравствуйте! Можете проверить статус моих заказов? Моя почта user_290c77bc52@olist-mail.com"}
    ],
    tools=tools,
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_f1J5W92Y9Wp6uJvlLC1stDCD', function=Function(arguments='{"customer_email":"user_290c77bc52@olist-mail.com"}', name='find_order'), type='function', index=0)], reasoning=None)


In [14]:
class CheckOrderStatus(BaseModel):
    """
    Аргументы для поиска информации о заказах в базе данных Olist.
    Используется агентом поддержки, когда клиенту нужно узнать статус заказа, 
    проверить детали или найти недавние покупки.
    """
    customer_email: Optional[str] = Field(
        default=None, 
        description="Email клиента, по которому нужно найти все его заказы в базе."
    )
    order_id: Optional[str] = Field(
        default=None,
        description="Уникальный идентификатор конкретного заказа (order_id), если клиент его назвал."
    )
    order_status: Optional[str] = Field(
        default=None,
        description="Статус заказа для фильтрации (например: shipped, delivered, processing, canceled)."
    )

In [15]:
def check_order_status(args: CheckOrderStatus) -> str:
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
       
        query = """
            SELECT 
                o.order_id, 
                o.order_status, 
                o.order_purchase_timestamp, 
                c.customer_email
            FROM olist_orders o 
            JOIN olist_customers c ON o.customer_id = c.customer_id
        """
        params = []
        conditions = []

        if args.order_id:
            conditions.append("o.order_id = ?")
            params.append(args.order_id)
            
        if args.customer_email:
            # Ищем всех покупателей с таким email через связку по customer_unique_id,
            # чтобы подтянуть абсолютно все их заказы со всех customer_id
            conditions.append("""
                c.customer_unique_id IN (
                    SELECT DISTINCT customer_unique_id 
                    FROM olist_customers 
                    WHERE customer_email = ?
                )
            """)
            params.append(args.customer_email)
            
        if args.order_status:
            conditions.append("o.order_status = ?")
            params.append(args.order_status)

        if conditions:
            query += " WHERE " + " AND ".join(conditions)

        # Сортируем от самых свежих заказов к старым и ограничиваем выдачу
        query += " ORDER BY o.order_purchase_timestamp DESC LIMIT 5"

        result = con.execute(query, params).fetchall()
        
        if not result:
            return "По вашему запросу заказов не найдено."

        output = []
        for row in result:
            output.append(f"• Заказ ID: {row[0]} | Статус: {row[1]} | Дата: {row[2]}")
            
        return "\n".join(output)
        
    finally:
    
        con.close()